# 13. Exportação dos artefatos de upload para Hugging Face

Este notebook exporta os artefatos gerados pelos notebooks de upload para o Hugging Face Hub:

```text
11_upload_adapters_to_huggingface.ipynb
12_upload_dataset_to_huggingface.ipynb
```

O objetivo não é reenviar arquivos para o Hugging Face. Este notebook apenas organiza, indexa e compacta os metadados locais produzidos durante os uploads, como manifestos, logs, cards gerados, resumos e arquivos de auditoria.

Ele foi criado separado do notebook `10_export_experiment_artifacts.ipynb` porque os notebooks 11 e 12 pertencem a uma etapa posterior: publicação dos adaptadores e do dataset no Hub. Assim, a exportação principal do experimento continua separada da exportação dos artefatos de publicação.

## 1. Política de espaço

Este notebook segue a mesma preocupação de espaço adotada nos notebooks de exportação anteriores.

A regra principal é:

```text
Não copiar adaptadores, datasets, caches ou pesos de modelo para exports/.
```

A pasta `exports/` será usada apenas para arquivos leves gerados por este próprio notebook, como:

```text
README.md
export_manifest.json
export_manifest.md
export_summary.json
index/file_index.csv
index/file_index.json
index/missing_optional_sources.json
index/skipped_files.json
archive_summary.json
```

O arquivo `.zip` final será criado lendo diretamente os arquivos originais de logs, manifestos e metadados dos notebooks 11 e 12. Esse zip será salvo fora da pasta `exports/`, em:

```text
/workspace/zip_files/
```

Essa estratégia evita duplicação pesada de arquivos. Mesmo que os notebooks 11 e 12 tenham produzido artefatos relacionados a adaptadores e datasets, este notebook não inclui os pesos dos adaptadores nem os arquivos grandes do dataset por padrão.

## 2. O que será exportado

Este notebook procura e exporta artefatos leves associados aos notebooks 11 e 12.

Para o notebook 11, são considerados artefatos como:

```text
exports/huggingface_upload/
logs/huggingface_upload/
manifests/huggingface_upload/
```

Para o notebook 12, são considerados artefatos como:

```text
exports/huggingface_dataset_upload/
logs/huggingface_dataset_upload/
manifests/huggingface_dataset_upload/
```

Essas pastas normalmente contêm arquivos como:

```text
README.md
model card
dataset card
experiment_adapters_manifest.json
dataset_upload_manifest.json
adapter_loading_examples.json
logs de upload
resumos de execução
manifestos finais
```

Se alguma dessas pastas não existir, isso não será tratado como erro fatal. O notebook registrará a ausência em `missing_optional_sources.json`.

## 3. O que não será exportado

Por padrão, este notebook não exporta:

```text
adapters/
data/canonical/
data/views/
data/cache/
outputs/
.venv/
pesos de modelo
arquivos .safetensors
arquivos .bin
arquivos .pt ou .pth
arquivos .gguf
cache do Hugging Face
```

Essa decisão evita duplicar arquivos grandes e reduz o risco de incluir pesos, caches ou dados sensíveis por engano.

Os notebooks 11 e 12 podem ter usado esses arquivos como fonte para upload, mas este notebook exporta apenas os **artefatos de publicação**: logs, manifestos, cards e índices.

## 4. Imports e configuração global

Esta seção define caminhos, flags e utilitários básicos.

A raiz do projeto esperada é:

```text
/workspace/pi-defense-exp
```

A pasta de metadados locais desta exportação será:

```text
exports/huggingface_upload_artifacts/full/
```

O zip final será salvo em:

```text
/workspace/zip_files/huggingface_upload_artifacts_full.zip
```

In [1]:
import hashlib
import json
import math
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("/workspace/pi-defense-exp")
RUN_MODE = "full"

EXPORT_ROOT = PROJECT_ROOT / "exports" / "huggingface_upload_artifacts" / RUN_MODE
EXPORT_INDEX_DIR = EXPORT_ROOT / "index"
ZIP_FILES_DIR = PROJECT_ROOT.parent / "zip_files"

ARCHIVE_PATH = ZIP_FILES_DIR / f"huggingface_upload_artifacts_{RUN_MODE}.zip"
ARCHIVE_INFO_PATH = ZIP_FILES_DIR / f"huggingface_upload_artifacts_{RUN_MODE}_zip_info.json"

MAX_FILE_SIZE_MB = 50

EXCLUDED_DIR_NAMES = {
    ".git",
    ".venv",
    "__pycache__",
    ".ipynb_checkpoints",
    "cache",
    "hub",
    "datasets",
    "transformers",
}

EXCLUDED_FILE_SUFFIXES = {
    ".safetensors",
    ".bin",
    ".pt",
    ".pth",
    ".ckpt",
    ".gguf",
    ".onnx",
    ".arrow",
}

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_INDEX_DIR.mkdir(parents=True, exist_ok=True)
ZIP_FILES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Export root:", EXPORT_ROOT)
print("Zip files dir:", ZIP_FILES_DIR)
print("Archive path:", ARCHIVE_PATH)

Project root: /workspace/pi-defense-exp
Export root: /workspace/pi-defense-exp/exports/huggingface_upload_artifacts/full
Zip files dir: /workspace/zip_files
Archive path: /workspace/zip_files/huggingface_upload_artifacts_full.zip


## 5. Funções utilitárias

As funções abaixo são usadas para escrever JSON válido, calcular hashes, indexar arquivos e evitar problemas com valores não serializáveis.

O notebook também trata `NaN`, `Infinity` e `-Infinity` como strings antes de salvar JSON. Isso evita arquivos JSON inválidos.

In [2]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sanitize_json_value(value):
    if isinstance(value, dict):
        return {str(k): sanitize_json_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_json_value(item) for item in value]
    if isinstance(value, tuple):
        return [sanitize_json_value(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
    return value


def write_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            sanitize_json_value(data),
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def is_excluded_path(path: Path) -> bool:
    parts = set(path.parts)
    if parts.intersection(EXCLUDED_DIR_NAMES):
        return True
    if path.suffix.lower() in EXCLUDED_FILE_SUFFIXES:
        return True
    return False


def should_skip_file(path: Path) -> tuple[bool, str | None]:
    if is_excluded_path(path):
        return True, "excluded_by_name_or_suffix"

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_FILE_SIZE_MB:
        return True, f"file_too_large_gt_{MAX_FILE_SIZE_MB}_mb"

    return False, None

## 6. Plano de fontes

Esta seção define quais pastas dos notebooks 11 e 12 serão consideradas.

As fontes são intencionalmente restritas a logs, manifestos e metadados de upload. O notebook não percorre `adapters/` nem `data/` para evitar incluir arquivos grandes.

In [3]:
SOURCE_PLAN = {
    "model_upload_exports": {
        "source": PROJECT_ROOT / "exports" / "huggingface_upload",
        "archive_root": "huggingface_upload_artifacts/model_upload/exports",
        "required": False,
        "description": "Metadados locais gerados pelo notebook 11 para upload dos adaptadores.",
    },
    "model_upload_logs": {
        "source": PROJECT_ROOT / "logs" / "huggingface_upload",
        "archive_root": "huggingface_upload_artifacts/model_upload/logs",
        "required": False,
        "description": "Logs locais do notebook 11.",
    },
    "model_upload_manifests": {
        "source": PROJECT_ROOT / "manifests" / "huggingface_upload",
        "archive_root": "huggingface_upload_artifacts/model_upload/manifests",
        "required": False,
        "description": "Manifestos locais do notebook 11.",
    },
    "dataset_upload_exports": {
        "source": PROJECT_ROOT / "exports" / "huggingface_dataset_upload",
        "archive_root": "huggingface_upload_artifacts/dataset_upload/exports",
        "required": False,
        "description": "Metadados locais gerados pelo notebook 12 para upload do dataset.",
    },
    "dataset_upload_logs": {
        "source": PROJECT_ROOT / "logs" / "huggingface_dataset_upload",
        "archive_root": "huggingface_upload_artifacts/dataset_upload/logs",
        "required": False,
        "description": "Logs locais do notebook 12.",
    },
    "dataset_upload_manifests": {
        "source": PROJECT_ROOT / "manifests" / "huggingface_dataset_upload",
        "archive_root": "huggingface_upload_artifacts/dataset_upload/manifests",
        "required": False,
        "description": "Manifestos locais do notebook 12.",
    },
}

source_plan_df = pd.DataFrame([
    {
        "name": name,
        "source": str(info["source"]),
        "archive_root": info["archive_root"],
        "required": info["required"],
        "exists": info["source"].exists(),
        "description": info["description"],
    }
    for name, info in SOURCE_PLAN.items()
])

display(source_plan_df)

,name,source,archive_root,required,exists,description
0,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/exports,False,True,Metadados locais gerados pelo notebook 11 para...
1,model_upload_logs,/workspace/pi-defense-exp/logs/huggingface_upload,huggingface_upload_artifacts/model_upload/logs,False,True,Logs locais do notebook 11.
2,model_upload_manifests,/workspace/pi-defense-exp/manifests/huggingfac...,huggingface_upload_artifacts/model_upload/mani...,False,True,Manifestos locais do notebook 11.
3,dataset_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/dataset_upload/ex...,False,True,Metadados locais gerados pelo notebook 12 para...
4,dataset_upload_logs,/workspace/pi-defense-exp/logs/huggingface_dat...,huggingface_upload_artifacts/dataset_upload/logs,False,True,Logs locais do notebook 12.
5,dataset_upload_manifests,/workspace/pi-defense-exp/manifests/huggingfac...,huggingface_upload_artifacts/dataset_upload/ma...,False,True,Manifestos locais do notebook 12.


## 7. Indexar artefatos existentes

Nesta etapa, o notebook percorre as fontes configuradas e cria uma lista dos arquivos que entrarão no zip.

Para cada arquivo incluído, são registrados:

```text
source_category
source_path
archive_path
size_bytes
sha256
```

Arquivos ausentes são registrados em `missing_optional_sources.json`. Arquivos ignorados por política são registrados em `skipped_files.json`.

In [4]:
file_records = []
missing_optional_sources = []
skipped_files = []

for source_name, info in SOURCE_PLAN.items():
    source_root = info["source"]
    archive_root = info["archive_root"]

    if not source_root.exists():
        missing_optional_sources.append({
            "source_name": source_name,
            "source_path": str(source_root),
            "reason": "source_missing",
            "required": info["required"],
            "description": info["description"],
        })
        continue

    if source_root.is_file():
        candidate_files = [source_root]
    else:
        candidate_files = [path for path in source_root.rglob("*") if path.is_file()]

    for path in candidate_files:
        skip, reason = should_skip_file(path)

        if skip:
            skipped_files.append({
                "source_name": source_name,
                "source_path": str(path),
                "reason": reason,
                "size_bytes": path.stat().st_size,
            })
            continue

        relative_path = path.relative_to(source_root)
        archive_path = str(Path(archive_root) / relative_path)

        file_records.append({
            "source_name": source_name,
            "source_path": str(path),
            "archive_path": archive_path,
            "size_bytes": path.stat().st_size,
            "size_mb": path.stat().st_size / (1024 * 1024),
            "sha256": file_sha256(path),
        })

file_index_df = pd.DataFrame(file_records)
missing_optional_sources_df = pd.DataFrame(missing_optional_sources)
skipped_files_df = pd.DataFrame(skipped_files)

print("Arquivos indexados para exportação:", len(file_index_df))
print("Fontes opcionais ausentes:", len(missing_optional_sources_df))
print("Arquivos ignorados por política:", len(skipped_files_df))

display(file_index_df.head(20))

Arquivos indexados para exportação: 16
Fontes opcionais ausentes: 0
Arquivos ignorados por política: 0


,source_name,source_path,archive_path,size_bytes,size_mb,sha256
0,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/expo...,2769,0.002641,4e49f48613fb69df227337d001e08786887fbe503bffd2...
1,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/expo...,5588,0.005329,4124a76c337b25a06b57f7856e1f7ae4ac8828e02f4e65...
2,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/expo...,2388,0.002277,a51e3ac42c43af48d86c49455c3a49be62659207a9c85e...
3,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/expo...,2815,0.002685,548595180bfca22bce046dddfaff7ecfabca9355fd9658...
4,model_upload_logs,/workspace/pi-defense-exp/logs/huggingface_upl...,huggingface_upload_artifacts/model_upload/logs...,22405,0.021367,97ebcbd6117a12ecd952647dcc3615dad158b08fa08c97...
5,model_upload_manifests,/workspace/pi-defense-exp/manifests/huggingfac...,huggingface_upload_artifacts/model_upload/mani...,9355,0.008922,25566a6222a779f7d7615e71ad71d4a73a393c5a4019cf...
6,model_upload_manifests,/workspace/pi-defense-exp/manifests/huggingfac...,huggingface_upload_artifacts/model_upload/mani...,2376,0.002266,261af8b7fa65b091da7063c2e343c659f96eca2c6e19b8...
7,dataset_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/dataset_upload/ex...,4148,0.003956,2ae684122975076c96df3d4000b2ed3bb9310505eb451d...
8,dataset_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/dataset_upload/ex...,3221,0.003072,c3a4eef7bb41c2917d3117e6468c8f52bbaf8eb9e02ad3...
9,dataset_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/dataset_upload/ex...,5617,0.005357,303a77f21bbde7ba010605cc8f0cd0817a2151f7e622db...


## 8. Gerar metadados locais da exportação

Agora serão criados arquivos leves dentro de `exports/huggingface_upload_artifacts/full/`.

Esses arquivos documentam a exportação, mas não duplicam os artefatos grandes do projeto.

In [5]:
export_summary = {
    "notebook": "13_export_huggingface_upload_artifacts",
    "created_at_utc": utc_now(),
    "project_root": str(PROJECT_ROOT),
    "run_mode": RUN_MODE,
    "export_root": str(EXPORT_ROOT),
    "archive_path": str(ARCHIVE_PATH),
    "max_file_size_mb": MAX_FILE_SIZE_MB,
    "total_files_indexed": len(file_records),
    "total_size_mb": sum(record["size_mb"] for record in file_records),
    "missing_optional_sources": len(missing_optional_sources),
    "skipped_files": len(skipped_files),
    "policy": {
        "no_intermediate_copy_of_large_artifacts": True,
        "zip_reads_from_original_sources": True,
        "exclude_model_weights_and_adapters": True,
        "exclude_dataset_cache": True,
    },
}

readme_text = f"""# Hugging Face Upload Artifacts Export

This export contains lightweight artifacts generated by the Hugging Face upload notebooks:

- `11_upload_adapters_to_huggingface.ipynb`
- `12_upload_dataset_to_huggingface.ipynb`

The export includes metadata, logs, manifests, cards, and audit files related to model adapter and dataset publication.

## Space policy

This export does not copy model adapters, dataset files, caches, or model weights into `exports/`.

The final zip is created by reading files directly from their original locations and is saved outside the export directory:

```text
{ARCHIVE_PATH}
```

## Included source categories

"""

for source_name, info in SOURCE_PLAN.items():
    readme_text += f"- `{source_name}`: `{info['source']}`\n"

readme_text += f"""

## Summary

- Created at UTC: `{export_summary['created_at_utc']}`
- Indexed files: `{export_summary['total_files_indexed']}`
- Total indexed size MB: `{export_summary['total_size_mb']:.4f}`
- Missing optional sources: `{export_summary['missing_optional_sources']}`
- Skipped files: `{export_summary['skipped_files']}`

## Important

This package is intended to document the Hugging Face publication process. It does not contain the full base model, full dataset cache, or adapter weights unless those files were incorrectly placed inside the upload metadata folders and passed the size/type filters.
"""

write_text(EXPORT_ROOT / "README.md", readme_text)
write_json(EXPORT_ROOT / "export_summary.json", export_summary)
write_json(EXPORT_INDEX_DIR / "file_index.json", {"files": file_records})
write_json(EXPORT_INDEX_DIR / "missing_optional_sources.json", {"missing_optional_sources": missing_optional_sources})
write_json(EXPORT_INDEX_DIR / "skipped_files.json", {"skipped_files": skipped_files})

file_index_csv_path = EXPORT_INDEX_DIR / "file_index.csv"
file_index_df.to_csv(file_index_csv_path, index=False)

print("Metadados leves criados em:", EXPORT_ROOT)

Metadados leves criados em: /workspace/pi-defense-exp/exports/huggingface_upload_artifacts/full


## 9. Criar manifesto da exportação

O manifesto registra o que foi considerado, o que entrou no pacote e o que ficou de fora.

Ele é salvo em JSON e Markdown.

In [6]:
def dataframe_to_markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "_Tabela vazia._"

    columns = [str(column) for column in df.columns]
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]

    for _, row in df.iterrows():
        values = []
        for column in df.columns:
            value = row[column]
            if pd.isna(value):
                value = ""
            elif isinstance(value, float):
                value = f"{value:.6f}"
            else:
                value = str(value)
            value = value.replace("|", "\\|")
            values.append(value)
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)

source_table_df = pd.DataFrame([
    {
        "source_name": name,
        "source_path": str(info["source"]),
        "exists": info["source"].exists(),
        "archive_root": info["archive_root"],
        "required": info["required"],
    }
    for name, info in SOURCE_PLAN.items()
])

manifest = {
    "notebook": "13_export_huggingface_upload_artifacts",
    "created_at_utc": utc_now(),
    "project_root": str(PROJECT_ROOT),
    "export_root": str(EXPORT_ROOT),
    "archive_path": str(ARCHIVE_PATH),
    "source_plan": sanitize_json_value(SOURCE_PLAN),
    "summary": export_summary,
    "files": file_records,
    "missing_optional_sources": missing_optional_sources,
    "skipped_files": skipped_files,
}

write_json(EXPORT_ROOT / "export_manifest.json", manifest)

manifest_md = f"""# Manifesto — Hugging Face Upload Artifacts Export

## Identificação

- Notebook: `13_export_huggingface_upload_artifacts`
- Gerado em UTC: `{manifest['created_at_utc']}`
- Projeto: `{PROJECT_ROOT}`
- Export root: `{EXPORT_ROOT}`
- Zip final: `{ARCHIVE_PATH}`

## Política

Este notebook não copia adaptadores, datasets, caches ou pesos de modelo para `exports/`.

O zip final é criado lendo diretamente os arquivos leves de logs, manifestos e metadados produzidos pelos notebooks 11 e 12.

## Fontes consideradas

{dataframe_to_markdown_table(source_table_df)}

## Resumo

| Campo | Valor |
|---|---:|
| Arquivos indexados | {len(file_records)} |
| Tamanho total indexado MB | {export_summary['total_size_mb']:.4f} |
| Fontes opcionais ausentes | {len(missing_optional_sources)} |
| Arquivos ignorados por política | {len(skipped_files)} |

## Arquivos de índice

- `index/file_index.csv`
- `index/file_index.json`
- `index/missing_optional_sources.json`
- `index/skipped_files.json`

## Observação

Fontes opcionais ausentes não indicam necessariamente erro. Elas indicam apenas que o notebook procurou artefatos esperados, mas eles ainda não foram gerados ou foram salvos em outro local.
"""

write_text(EXPORT_ROOT / "export_manifest.md", manifest_md)

print("Manifesto criado:")
print(EXPORT_ROOT / "export_manifest.json")
print(EXPORT_ROOT / "export_manifest.md")

Manifesto criado:
/workspace/pi-defense-exp/exports/huggingface_upload_artifacts/full/export_manifest.json
/workspace/pi-defense-exp/exports/huggingface_upload_artifacts/full/export_manifest.md


## 10. Adicionar metadados gerados por este notebook ao índice

Os próprios arquivos gerados nesta exportação também devem entrar no zip.

Esses arquivos são leves e ficam em `exports/huggingface_upload_artifacts/full/`.

In [7]:
metadata_records = []

for path in EXPORT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    skip, reason = should_skip_file(path)
    if skip:
        skipped_files.append({
            "source_name": "generated_export_metadata",
            "source_path": str(path),
            "reason": reason,
            "size_bytes": path.stat().st_size,
        })
        continue

    archive_path = str(
        Path("huggingface_upload_artifacts")
        / RUN_MODE
        / "export_metadata"
        / path.relative_to(EXPORT_ROOT)
    )

    metadata_records.append({
        "source_name": "generated_export_metadata",
        "source_path": str(path),
        "archive_path": archive_path,
        "size_bytes": path.stat().st_size,
        "size_mb": path.stat().st_size / (1024 * 1024),
        "sha256": file_sha256(path),
    })

all_archive_records = file_records + metadata_records

all_archive_df = pd.DataFrame(all_archive_records)
all_archive_df.to_csv(EXPORT_INDEX_DIR / "archive_file_index.csv", index=False)
write_json(EXPORT_INDEX_DIR / "archive_file_index.json", {"files": all_archive_records})

print("Arquivos originais:", len(file_records))
print("Metadados gerados:", len(metadata_records))
print("Total no zip:", len(all_archive_records))

display(all_archive_df.head(20))

Arquivos originais: 16
Metadados gerados: 11
Total no zip: 27


,source_name,source_path,archive_path,size_bytes,size_mb,sha256
0,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/expo...,2769,0.002641,4e49f48613fb69df227337d001e08786887fbe503bffd2...
1,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/expo...,5588,0.005329,4124a76c337b25a06b57f7856e1f7ae4ac8828e02f4e65...
2,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/expo...,2388,0.002277,a51e3ac42c43af48d86c49455c3a49be62659207a9c85e...
3,model_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/model_upload/expo...,2815,0.002685,548595180bfca22bce046dddfaff7ecfabca9355fd9658...
4,model_upload_logs,/workspace/pi-defense-exp/logs/huggingface_upl...,huggingface_upload_artifacts/model_upload/logs...,22405,0.021367,97ebcbd6117a12ecd952647dcc3615dad158b08fa08c97...
5,model_upload_manifests,/workspace/pi-defense-exp/manifests/huggingfac...,huggingface_upload_artifacts/model_upload/mani...,9355,0.008922,25566a6222a779f7d7615e71ad71d4a73a393c5a4019cf...
6,model_upload_manifests,/workspace/pi-defense-exp/manifests/huggingfac...,huggingface_upload_artifacts/model_upload/mani...,2376,0.002266,261af8b7fa65b091da7063c2e343c659f96eca2c6e19b8...
7,dataset_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/dataset_upload/ex...,4148,0.003956,2ae684122975076c96df3d4000b2ed3bb9310505eb451d...
8,dataset_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/dataset_upload/ex...,3221,0.003072,c3a4eef7bb41c2917d3117e6468c8f52bbaf8eb9e02ad3...
9,dataset_upload_exports,/workspace/pi-defense-exp/exports/huggingface_...,huggingface_upload_artifacts/dataset_upload/ex...,5617,0.005357,303a77f21bbde7ba010605cc8f0cd0817a2151f7e622db...


## 11. Criar arquivo compactado

O zip é criado diretamente a partir dos arquivos originais indexados, sem copiar previamente os artefatos para `exports/`.

O arquivo final será salvo em:

```text
/workspace/zip_files/
```

In [8]:
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

if not all_archive_records:
    raise RuntimeError("Nenhum arquivo foi indexado para compactação.")

with zipfile.ZipFile(
    ARCHIVE_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zip_file:
    for record in all_archive_records:
        source_path = Path(record["source_path"])
        archive_path = record["archive_path"]

        if not source_path.exists():
            raise FileNotFoundError(f"Arquivo indexado não existe mais: {source_path}")

        zip_file.write(source_path, archive_path)

archive_summary = {
    "created_at_utc": utc_now(),
    "archive_path": str(ARCHIVE_PATH),
    "archive_size_bytes": ARCHIVE_PATH.stat().st_size,
    "archive_size_mb": ARCHIVE_PATH.stat().st_size / (1024 * 1024),
    "files_in_archive": len(all_archive_records),
    "archive_sha256": file_sha256(ARCHIVE_PATH),
    "source_export_root": str(EXPORT_ROOT),
}

write_json(EXPORT_ROOT / "archive_summary.json", archive_summary)
write_json(ARCHIVE_INFO_PATH, archive_summary)

print("Zip criado:", ARCHIVE_PATH)
print(f"Arquivos incluídos: {archive_summary['files_in_archive']}")
print(f"Tamanho do zip: {archive_summary['archive_size_mb']:.4f} MB")
print("SHA256:", archive_summary["archive_sha256"])

Zip criado: /workspace/zip_files/huggingface_upload_artifacts_full.zip
Arquivos incluídos: 27
Tamanho do zip: 0.0312 MB
SHA256: ee3964b317506b43fd4af29fc63543682d81e042d5b1ff1a5d816ae109304845


## 12. Conferência final

A última etapa verifica se os principais arquivos de exportação foram criados.

In [9]:
expected_outputs = {
    "readme": EXPORT_ROOT / "README.md",
    "export_summary": EXPORT_ROOT / "export_summary.json",
    "export_manifest_json": EXPORT_ROOT / "export_manifest.json",
    "export_manifest_md": EXPORT_ROOT / "export_manifest.md",
    "file_index_csv": EXPORT_INDEX_DIR / "file_index.csv",
    "file_index_json": EXPORT_INDEX_DIR / "file_index.json",
    "archive_file_index_csv": EXPORT_INDEX_DIR / "archive_file_index.csv",
    "archive_file_index_json": EXPORT_INDEX_DIR / "archive_file_index.json",
    "missing_optional_sources": EXPORT_INDEX_DIR / "missing_optional_sources.json",
    "skipped_files": EXPORT_INDEX_DIR / "skipped_files.json",
    "archive_summary": EXPORT_ROOT / "archive_summary.json",
    "archive": ARCHIVE_PATH,
    "archive_info": ARCHIVE_INFO_PATH,
}

final_check_rows = []

for name, path in expected_outputs.items():
    final_check_rows.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else None,
    })

final_check_df = pd.DataFrame(final_check_rows)
display(final_check_df)

if not final_check_df["exists"].all():
    missing = final_check_df.loc[~final_check_df["exists"], "name"].tolist()
    raise RuntimeError("Arquivos esperados não foram criados: " + ", ".join(missing))

print("Exportação dos artefatos de upload Hugging Face concluída com sucesso.")
print("Zip final:", ARCHIVE_PATH)

,name,path,exists,size_bytes
0,readme,/workspace/pi-defense-exp/exports/huggingface_...,True,1656
1,export_summary,/workspace/pi-defense-exp/exports/huggingface_...,True,700
2,export_manifest_json,/workspace/pi-defense-exp/exports/huggingface_...,True,9711
3,export_manifest_md,/workspace/pi-defense-exp/exports/huggingface_...,True,2182
4,file_index_csv,/workspace/pi-defense-exp/exports/huggingface_...,True,4626
5,file_index_json,/workspace/pi-defense-exp/exports/huggingface_...,True,6870
6,archive_file_index_csv,/workspace/pi-defense-exp/exports/huggingface_...,True,7707
7,archive_file_index_json,/workspace/pi-defense-exp/exports/huggingface_...,True,11524
8,missing_optional_sources,/workspace/pi-defense-exp/exports/huggingface_...,True,36
9,skipped_files,/workspace/pi-defense-exp/exports/huggingface_...,True,25


Exportação dos artefatos de upload Hugging Face concluída com sucesso.
Zip final: /workspace/zip_files/huggingface_upload_artifacts_full.zip


## 13. Próximos passos

Depois de executar este notebook, o arquivo compactado em `/workspace/zip_files/` pode ser usado para arquivar ou compartilhar os artefatos de publicação no Hugging Face.

Esse pacote não substitui os repositórios remotos no Hub. Ele serve como registro local do processo de upload, incluindo cards, manifestos, logs e índices.

Antes de compartilhar o pacote, revise especialmente:

```text
index/file_index.csv
index/skipped_files.json
index/missing_optional_sources.json
```

Isso ajuda a confirmar que nenhum arquivo grande, token, cache ou peso de modelo foi incluído por engano.